# Reinforcement Learning for Language Models: From Policy Gradient to PPO

The full code of [Reinforcement Learning for Language Models: From Policy Gradient to PPO](https://residual-stream.lvtan71.workers.dev/posts/policy-gradient-to-ppo/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/policy-gradient-to-ppo) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/policy-gradient-to-ppo.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q numpy "torch==2.14.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "policy-gradient-to-ppo"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `tiny_policy.py`

A table policy that writes 3 tokens from a vocabulary of 8, rewarded for matching a hidden target: REINFORCE with and without the reward offset, REINFORCE with a batch-mean baseline, the baseline with 4 passes per batch, and a simplified PPO, 10 seeds each at two Adam learning rates. Then one batch's gradient against the exact gradient over all 512 sequences.

In [ ]:
!{sys.executable} tiny_policy.py

The post printed:

```text
Adam learning rate 0.1: mean matches per sequence (of 3) over 10 seeds, at update
                            10     50    100    200    300   seeds solved (>= 2.9 at the end)
REINFORCE                 0.37   0.38   0.53   1.07   1.51   0 of 10
REINFORCE, no offset      0.46   1.76   2.48   2.64   2.78   8 of 10
REINFORCE + baseline      0.47   2.28   2.97   2.99   2.99   10 of 10
baseline, 4 passes        1.11   2.63   2.70   2.70   2.70   7 of 10
PPO                       0.79   2.69   2.69   2.80   2.90   9 of 10
PPO's clip was active for 3% of sequences in passes 2 to 4
Adam learning rate 0.03: mean matches per sequence (of 3) over 10 seeds, at update
                            10     50    100    200    300   seeds solved (>= 2.9 at the end)
REINFORCE                 0.36   0.37   0.41   0.49   0.66   0 of 10
REINFORCE, no offset      0.38   0.78   1.50   2.63   2.89   5 of 10
REINFORCE + baseline      0.38   0.86   1.92   2.91   2.95   10 of 10
baseline, 4 passes        0.51   2.69   2.97   2.99   2.99   10 of 10
PPO                       0.48   2.57   2.96   2.99   2.99   10 of 10
PPO's clip was active for 2% of sequences in passes 2 to 4
one batch's gradient vs the exact one, rewards 10 to 13    : cosine 0.02
one batch's gradient vs the exact one, rewards 0 to 3      : cosine 0.40
one batch's gradient vs the exact one, minus the batch mean: cosine 0.47
```